# 03 - Análise Social: Renda por Bairro em Recife

Agrega os setores censitários por bairro, calcula renda média, % abaixo da linha de pobreza e gera mapa coroplético interativo.

In [ ]:
import os
import json
from pathlib import Path
import pandas as pd
import geopandas as gpd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / '.git').exists():
            return p
    return Path.cwd()

ROOT       = find_root()
INPUT_PATH = ROOT / 'data' / 'processed' / 'recife_renda.geojson'
OUTPUT_CSV = ROOT / 'data' / 'processed' / 'recife_renda_bairros.csv'

print(f'ROOT    : {ROOT}')
print(f'Entrada : {INPUT_PATH}')
print(f'Saída   : {OUTPUT_CSV}')

In [ ]:
# Carrega o GeoJSON com setores censitários e dados de renda
gdf = gpd.read_file(INPUT_PATH)

COLS_NUM = ['V06001', 'V06002', 'V06003', 'V06004', 'V06005', 'V06006']
for col in COLS_NUM:
    gdf[col] = pd.to_numeric(gdf[col], errors='coerce')

print(f'Setores carregados  : {len(gdf)}')
print(f'Bairros únicos      : {gdf["NM_BAIRRO"].nunique()}')
print()
# Dicionário das variáveis utilizadas
print('Variáveis:')
print('  V06001 – responsáveis por domicílio (total do setor)')
print('  V06003 – média de moradores por domicílio')
print('  V06004 – rendimento médio mensal do responsável (R$)')
print('  V06006 – rendimento mediano mensal do responsável (R$)')

display(gdf[['CD_SETOR', 'NM_BAIRRO', 'V06001', 'V06003', 'V06004', 'V06006']].head())

## 1 – Renda média por bairro

Média ponderada de `V06004` (renda média do responsável) usando `V06001` (total de responsáveis do setor) como peso.

In [ ]:
gdf['renda_pond'] = gdf['V06004'] * gdf['V06001']

agg = (
    gdf
    .groupby(['CD_BAIRRO', 'NM_BAIRRO'], dropna=False)
    .agg(
        n_setores          = ('CD_SETOR',    'count'),
        total_responsaveis = ('V06001',      'sum'),
        soma_pond          = ('renda_pond',  'sum'),
        mediana_renda      = ('V06006',      'median'),
    )
    .reset_index()
)

agg['renda_media'] = (agg['soma_pond'] / agg['total_responsaveis']).round(2)
agg = agg.drop(columns=['soma_pond'])

print(f'Total de bairros: {len(agg)}')
display(agg.sort_values('renda_media', ascending=False).head(10))

## 2 – % de pessoas abaixo da linha de pobreza

**Metodologia:** estimativa de renda per capita do setor = `V06004 / V06003` (renda média ÷ moradores/domicílio).  
Setores com renda per capita < R$ 436/mês são classificados como *em situação de pobreza* (critério Bolsa Família 2022).  
O % por bairro é calculado como a fração de responsáveis nesses setores sobre o total do bairro.

In [ ]:
LINHA_POBREZA = 436.0  # R$/mês per capita — critério Bolsa Família 2022

gdf['renda_percapita'] = gdf['V06004'] / gdf['V06003']
gdf['em_pobreza']      = gdf['renda_percapita'] < LINHA_POBREZA
gdf['resp_em_pobreza'] = gdf['em_pobreza'].astype(float) * gdf['V06001']

agg_pob = (
    gdf
    .groupby(['CD_BAIRRO', 'NM_BAIRRO'], dropna=False)
    .agg(
        resp_pobreza = ('resp_em_pobreza', 'sum'),
        total_resp   = ('V06001',          'sum'),
    )
    .reset_index()
)

agg_pob['pct_pobreza'] = (agg_pob['resp_pobreza'] / agg_pob['total_resp'] * 100).round(1)

agg = agg.merge(
    agg_pob[['NM_BAIRRO', 'resp_pobreza', 'pct_pobreza']],
    on='NM_BAIRRO',
    how='left'
)

print(f'Linha de pobreza: R$ {LINHA_POBREZA:.0f}/mês per capita')
print(f'Bairros com > 0% em pobreza: {(agg["pct_pobreza"] > 0).sum()}')
display(agg.sort_values('pct_pobreza', ascending=False).head(10))

## 3 – Geometria por bairro

Dissolve os polígonos de setores para obter um polígono único por bairro.

In [ ]:
geo_bairro = (
    gdf[['NM_BAIRRO', 'geometry']]
    .dissolve(by='NM_BAIRRO')
    .reset_index()
    .to_crs(epsg=4326)
)

geo_bairro = geo_bairro.merge(agg, on='NM_BAIRRO', how='left')

print(f'Polígonos de bairro: {len(geo_bairro)}')
display(geo_bairro[['NM_BAIRRO', 'n_setores', 'total_responsaveis', 'renda_media', 'pct_pobreza']].head())

## 4 – Mapa coroplético de renda por bairro

In [ ]:
geojson = json.loads(geo_bairro.to_json())
for feat in geojson['features']:
    feat['id'] = feat['properties']['NM_BAIRRO']

fig = px.choropleth_mapbox(
    geo_bairro,
    geojson=geojson,
    locations='NM_BAIRRO',
    featureidkey='properties.NM_BAIRRO',
    color='renda_media',
    color_continuous_scale='RdYlGn',
    range_color=[geo_bairro['renda_media'].quantile(0.05), geo_bairro['renda_media'].quantile(0.95)],
    mapbox_style='carto-positron',
    zoom=11,
    center={'lat': -8.05, 'lon': -34.92},
    opacity=0.75,
    hover_name='NM_BAIRRO',
    hover_data={
        'NM_BAIRRO':          False,
        'renda_media':        ':,.0f',
        'mediana_renda':      ':,.0f',
        'pct_pobreza':        ':.1f',
        'total_responsaveis': ':,.0f',
    },
    labels={
        'renda_media':        'Renda Média (R$)',
        'mediana_renda':      'Mediana Renda (R$)',
        'pct_pobreza':        '% em Pobreza',
        'total_responsaveis': 'Total Responsáveis',
    },
    title='Renda Média do Responsável por Bairro — Recife (Censo 2022)',
)

fig.update_layout(
    margin=dict(l=0, r=0, t=50, b=0),
    height=620,
    coloraxis_colorbar=dict(title='Renda Média<br>(R$)'),
    modebar_add=['zoomInMapbox', 'zoomOutMapbox', 'resetViewMapbox'],
)

fig.show(config={'scrollZoom': True})

## 5 – Top 10 bairros mais ricos e mais pobres

In [ ]:
COLS_EXIBIR = ['NM_BAIRRO', 'renda_media', 'mediana_renda', 'pct_pobreza', 'total_responsaveis']

top10_ricos  = agg.nlargest(10,  'renda_media')[COLS_EXIBIR].reset_index(drop=True)
top10_pobres = agg.nsmallest(10, 'renda_media')[COLS_EXIBIR].reset_index(drop=True)

top10_ricos.index  += 1
top10_pobres.index += 1

print('TOP 10 — BAIRROS MAIS RICOS (renda média do responsável):')
display(
    top10_ricos.style
    .format({'renda_media': 'R$ {:,.0f}', 'mediana_renda': 'R$ {:,.0f}', 'pct_pobreza': '{:.1f}%', 'total_responsaveis': '{:,.0f}'})
    .background_gradient(subset=['renda_media'], cmap='Greens')
)

print('\nTOP 10 — BAIRROS MAIS POBRES (renda média do responsável):')
display(
    top10_pobres.style
    .format({'renda_media': 'R$ {:,.0f}', 'mediana_renda': 'R$ {:,.0f}', 'pct_pobreza': '{:.1f}%', 'total_responsaveis': '{:,.0f}'})
    .background_gradient(subset=['renda_media'], cmap='Reds_r')
)

In [ ]:
fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Top 10 Mais Ricos', 'Top 10 Mais Pobres'],
    horizontal_spacing=0.25,
)

fig2.add_trace(
    go.Bar(
        x=top10_ricos['renda_media'],
        y=top10_ricos['NM_BAIRRO'],
        orientation='h',
        marker_color='#27ae60',
        name='Mais Ricos',
        text=top10_ricos['renda_media'].apply(lambda x: f'R$ {x:,.0f}'),
        textposition='outside',
    ),
    row=1, col=1,
)

fig2.add_trace(
    go.Bar(
        x=top10_pobres['renda_media'],
        y=top10_pobres['NM_BAIRRO'],
        orientation='h',
        marker_color='#c0392b',
        name='Mais Pobres',
        text=top10_pobres['renda_media'].apply(lambda x: f'R$ {x:,.0f}'),
        textposition='outside',
    ),
    row=1, col=2,
)

fig2.update_yaxes(autorange='reversed')
fig2.update_layout(
    height=420,
    showlegend=False,
    title='Desigualdade de Renda entre Bairros — Recife (Censo 2022)',
    margin=dict(l=10, r=10, t=60, b=10),
)

fig2.show()

## 6 – Salvar resultado em CSV

In [ ]:
antes = len(agg)
agg = agg.dropna(subset=['NM_BAIRRO']).reset_index(drop=True)
removidas = antes - len(agg)

print(f'Linhas antes da limpeza : {antes}')
print(f'Linhas removidas (NaN)  : {removidas}')
print(f'Linhas após limpeza     : {len(agg)}')
print()
print('Verificação de NaN no dataframe final:')
display(agg.isnull().sum().to_frame('qtd_nulos'))

In [ ]:
agg_out = agg[[
    'CD_BAIRRO', 'NM_BAIRRO',
    'n_setores', 'total_responsaveis',
    'renda_media', 'mediana_renda',
    'resp_pobreza', 'pct_pobreza',
]].copy()

agg_out.columns = [
    'cd_bairro', 'nm_bairro',
    'n_setores', 'total_responsaveis',
    'renda_media_responsavel', 'mediana_renda_responsavel',
    'responsaveis_em_pobreza', 'pct_abaixo_pobreza',
]

agg_out = agg_out.sort_values('renda_media_responsavel', ascending=False).reset_index(drop=True)

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
agg_out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'Arquivo salvo: {OUTPUT_CSV}')
print(f'Total de bairros: {len(agg_out)}')
print(f'Colunas: {agg_out.columns.tolist()}')
display(agg_out.head(10))

## 7 – Nota de 0 a 10 por bairro (normalização min-max)

A nota é calculada pela fórmula:

$$\text{nota} = \frac{\text{renda\_media} - \text{renda\_min}}{\text{renda\_max} - \text{renda\_min}} \times 10$$

Bairro com menor renda recebe **0,00** e o de maior renda recebe **10,00**.

In [ ]:
renda_min = agg['renda_media'].min()
renda_max = agg['renda_media'].max()

agg['nota_dimensao'] = ((agg['renda_media'] - renda_min) / (renda_max - renda_min) * 10).round(2)

print(f'Renda mínima : R$ {renda_min:,.2f}  → nota 0,00')
print(f'Renda máxima : R$ {renda_max:,.2f}  → nota 10,00')
print(f'Renda média  : R$ {agg["renda_media"].mean():,.2f}  → nota {((agg["renda_media"].mean() - renda_min) / (renda_max - renda_min) * 10):.2f}')

notas_exibir = (
    agg[['NM_BAIRRO', 'renda_media', 'nota_dimensao', 'pct_pobreza']]
    .sort_values('nota_dimensao', ascending=False)
    .reset_index(drop=True)
)
notas_exibir.index += 1

display(
    notas_exibir.style
    .format({
        'renda_media':    'R$ {:,.2f}',
        'nota_dimensao':  '{:.2f}',
        'pct_pobreza':    '{:.1f}%',
    })
    .background_gradient(subset=['nota_dimensao'], cmap='RdYlGn')
    .set_caption('Nota de Renda (0–10) por Bairro — Recife')
)

In [ ]:
fig3 = px.bar(
    notas_exibir.sort_values('nota_dimensao'),
    x='nota_dimensao',
    y='NM_BAIRRO',
    orientation='h',
    color='nota_dimensao',
    color_continuous_scale='RdYlGn',
    range_color=[0, 10],
    text='nota_dimensao',
    hover_data={'renda_media': ':,.2f', 'pct_pobreza': ':.1f'},
    labels={
        'nota_dimensao': 'Nota (0–10)',
        'NM_BAIRRO':     'Bairro',
        'renda_media':   'Renda Média (R$)',
        'pct_pobreza':   '% em Pobreza',
    },
    title='Índice de Renda por Bairro — Recife (Censo 2022)',
    height=1800,
)

fig3.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig3.update_layout(
    margin=dict(l=10, r=60, t=50, b=10),
    showlegend=False,
    coloraxis_showscale=False,
    xaxis=dict(range=[0, 11], title='Nota (0–10)'),
)

fig3.show()

## 8 – Salvar notas_renda.csv

In [ ]:
OUTPUT_NOTAS = os.path.join(ROOT, 'data', 'processed', 'notas_renda.csv')


def format_brl(valor: float) -> str:
    """Formata valor em reais no padrão brasileiro: R$ 1.234,56"""
    return f"R$ {valor:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.')


notas_renda = pd.DataFrame({
    'bairro':         agg['NM_BAIRRO'],
    'nota_dimensao':  agg['nota_dimensao'],
    'dado_principal': agg['renda_media'].apply(format_brl),
})

notas_renda = notas_renda.sort_values('nota_dimensao', ascending=False).reset_index(drop=True)

notas_renda.to_csv(OUTPUT_NOTAS, index=False, encoding='utf-8-sig')

print(f'Arquivo salvo: {OUTPUT_NOTAS}')
print(f'Total de bairros: {len(notas_renda)}')
display(notas_renda.head(10))
print('...')
display(notas_renda.tail(10))